# Connect and explore

- Confirm the workbench can read events from the platform
- See which data sets and fields are available
- Load events into Spark and query them with SQL

---
<sub>TechDetechtives analytics workbench. This notebook is a modified work derived from the HELK notebook `demos/read_elasticsearch_via_spark.ipynb` (https://github.com/Cyb3rWard0g/HELK), Copyright Roberto Rodriguez (@Cyb3rWard0g), licensed GPL-3.0. Modified by TechDetechtives, 2026: reads from the platform through a read-only client instead of a bundled Elasticsearch, uses ECS field names, and adds a sample-data mode. Licensed GPL-3.0; see `analytics/LICENSE`.</sub>

## Import the hunting helpers

In [ ]:
import pandas as pd
import td_hunt as td

pd.set_option("display.max_colwidth", 120)
print("td_hunt", td.__version__, "| demo mode:", td.demo_mode())

## Check the connection

In demo mode (no `TD_ES_HOST` in `config/techdetechtives.env`) every notebook runs on the bundled sample data. With a platform configured, this cell shows the Elasticsearch version and how many events the read-only key can see.

In [ ]:
if td.demo_mode():
    print("Demo mode: using bundled sample data.")
else:
    es = td.client()
    print("Elasticsearch", es.info()["version"]["number"])
    print("Events visible in logs-*:", es.count(index="logs-*", ignore_unavailable=True)["count"])

## Data sets

Named data sets map to the index patterns the platform writes to. Any index pattern can be passed to `td.events()` directly as well.

In [ ]:
pd.DataFrame([{"name": name, "index pattern": spec["index"]} for name, spec in td.DATASETS.items()])

## Load process events

`td.events()` returns a pandas DataFrame with one column per field, named as on the platform (`process.parent.name`, `host.name`, ...).

In [ ]:
process_df = td.events("process", since="24h")
process_df.shape

In [ ]:
process_df[["@timestamp", "host.name", "user.name", "process.parent.name", "process.name"]].head(10)

## Which hosts and programs are present?

In [ ]:
process_df.groupby(["host.name", "process.name"]).size().sort_values(ascending=False).head(15)

## Network connections

Repeated connections from one host to the same external address at a steady interval are worth a look. This counts connections per pair and measures how regular the gaps between them are (a low spread means machine-like timing).

In [ ]:
conn_df = td.events("zeek_conn", since="24h")
pairs = (
    conn_df.sort_values("@timestamp")
    .groupby(["source.ip", "destination.ip", "destination.port"])["@timestamp"]
    .agg(connections="count", gap_spread_seconds=lambda t: t.diff().dt.total_seconds().std())
    .reset_index()
)
pairs[pairs["connections"] >= 4].sort_values("gap_spread_seconds")

## Query with Spark SQL

`td.to_spark()` registers the events as a SQL view. Dots in field names become underscores (`process.parent.name` is `process_parent_name`), and `@timestamp` becomes `timestamp`.

In [ ]:
spark = td.spark_session()
td.to_spark(process_df, view="process", spark=spark)
spark.sql("""
    SELECT host_name, process_parent_name, process_name, COUNT(*) AS starts
    FROM process
    GROUP BY host_name, process_parent_name, process_name
    ORDER BY starts DESC
""").show(20, truncate=False)